# Explainable Multi-Objective Framework for Cybersecure BIM Software Design & Sustainable Smart Construction
### End-to-End Experimental Walkthrough and Demonstration

This notebook provides a fully reproducible demonstration of the three core pillars:
1. **Multi-Objective Sustainable Construction Optimization** (Embodied Carbon, Life Cycle Cost, Structural Safety)
2. **Zero-Trust Cybersecure BIM Architecture** (JWT Authentication, RBAC, Cryptographic Model Integrity)
3. **Cybersecurity Analytics & Explainable AI** (ToN_IoT ML Intrusion Detection, TreeSHAP Attributions)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.ingestion.ifc_parser import IFCParser
from src.ingestion.material_mapper import MaterialMapper
from src.sustainability.carbon_calculator import CarbonCalculator
from src.lifecycle_cost.lcc_calculator import LifeCycleCostCalculator
from src.structural_constraints.safety_engine import StructuralSafetyEngine
from src.optimization.multi_objective_engine import MultiObjectiveOptimizationEngine
from src.security_controls.zero_trust_engine import ZeroTrustEngine
from src.preprocessing.cyber_preprocessor import CyberDataPreprocessor
from src.models.cyber_models import CyberModelSuite
from src.explainability.xai_engine import ExplainableAIEngine

print("All framework modules imported successfully!")


## 1. BIM Model Parsing and Physical Element Extraction
Extract physical structural elements from the official buildingSMART IFC4 certification dataset.


In [ ]:
ifc_path = "data/raw/bim/Building-Structural.ifc"
parser = IFCParser(ifc_path)
raw_df = parser.parse_all_elements()
mapper = MaterialMapper()
enriched_df = mapper.enrich_bim_elements(raw_df)

print(f"IFC Schema: {parser.schema}")
print(f"Authoritative SHA-256 Hash: {parser.compute_model_hash()}")
print(f"Total Physical Elements: {len(enriched_df)}")
display(enriched_df[["guid", "ifc_class", "material_name", "volume_m3", "GWP_A1_A3_kgCO2eq", "basic_rate_inr"]].head(8))


## 2. Baseline Embodied Carbon and 30-Year Lifecycle Cost Assessment
Calculate Cradle-to-Grave carbon (EN 15804) and Net Present Value lifecycle costing (CPWD DSR 2021).


In [ ]:
c_calc = CarbonCalculator(transport_distance_km=50.0)
l_calc = LifeCycleCostCalculator(assessment_years=30, discount_rate=0.04)
s_engine = StructuralSafetyEngine()

carbon_res = c_calc.evaluate_building_carbon(enriched_df)
lcc_res = l_calc.evaluate_building_lcc(enriched_df)
safety_res = s_engine.evaluate_building_structural_safety(enriched_df)

print("=== BASELINE EVALUATION SUMMARY ===")
print(f"Total Embodied Carbon: {carbon_res['summary']['total_embodied_carbon_kgCO2eq']:,.1f} kg CO2 eq")
print(f"Initial Capital Cost: INR {lcc_res['summary']['initial_capital_cost_inr']:,.2f}")
print(f"30-Year Lifecycle Cost NPV: INR {lcc_res['summary']['total_lifecycle_cost_npv_inr']:,.2f}")
print(f"Structural Feasibility: {safety_res['is_structurally_feasible']} (Max DCR: {safety_res['max_demand_capacity_ratio']})")


## 3. Evolutionary Multi-Objective Optimization (NSGA-II)
Execute constrained optimization to find the non-dominated Pareto frontier balancing carbon, cost, and safety.


In [ ]:
opt_engine = MultiObjectiveOptimizationEngine(enriched_df, random_seed=42)
print("Running NSGA-II...")
nsga2_res = opt_engine.run_nsga2(pop_size=40, n_gen=25)

print(f"Completed in {nsga2_res['elapsed_seconds']}s | Feasibility: {nsga2_res['feasibility_rate_percent']}% | HV: {nsga2_res['hypervolume']}")
print("
--- Feasible Design Alternatives ---")
for alt_name, alt_data in nsga2_res["alternatives"].items():
    print(f"
[{alt_name.upper()}]")
    print(f"  Carbon: {alt_data['embodied_carbon_kgCO2eq']:,} kg CO2e | Cost: INR {alt_data['lifecycle_cost_npv_inr']:,}")
    print(f"  Beam: {alt_data['beam_material']} | Wall: {alt_data['wall_material']}")


## 4. Machine Learning Cybersecurity Intrusion Detection
Train and evaluate supervised intrusion detection on the ToN_IoT NetFlow v2 benchmark.


In [ ]:
prep = CyberDataPreprocessor(random_state=42)
ton = prep.preprocess_ton_iot(sample_size=15000)

suite = CyberModelSuite(random_state=42)
bin_results = suite.train_evaluate_binary(
    ton["X_train"], ton["y_bin_train"],
    ton["X_test"], ton["y_bin_test"]
)

results_df = pd.DataFrame.from_dict(bin_results, orient="index")
display(results_df[["model_name", "accuracy", "f1_score", "roc_auc", "false_positive_rate", "inference_latency_us"]])


## 5. Zero-Trust Access Control & Cryptographic Model Tamper Detection
Test NIST SP 800-207 Zero-Trust enforcement and verify model integrity against unauthorized mutation.


In [ ]:
zt = ZeroTrustEngine()
zt.register_model_baseline_hash("struct_v1", ifc_path)

# Test 1: Authorized engineer update
eng_token = zt.generate_token("alice_eng", "structural_engineer")
ok1, stat1, _ = zt.authorize_action(eng_token, "element:update", "IfcBeam_101")
print(f"Structural Engineer 'element:update': Allowed={ok1}, Status={stat1}")

# Test 2: Unauthorized viewer update (Privilege Escalation attempt)
viewer_token = zt.generate_token("bob_guest", "viewer")
ok2, stat2, _ = zt.authorize_action(viewer_token, "element:update", "IfcBeam_101")
print(f"Viewer 'element:update': Allowed={ok2}, Status={stat2}")

# Test 3: Cryptographic model integrity verification
int_ok, int_msg = zt.verify_model_integrity("struct_v1", ifc_path, "alice_eng")
print(f"Model File Integrity Check: OK={int_ok}, Message={int_msg}")


## 6. Explainable AI (TreeSHAP) Feature Attributions
Explain model predictions using Shapley additive explanations and verify explanation stability.


In [ ]:
xgb_model = suite.trained_models["XGB_binary"]
xai = ExplainableAIEngine(xgb_model, ton["feature_names"])

glob_attr = xai.compute_global_attributions(ton["X_test"][:200])
top_feats = pd.DataFrame(glob_attr["top_features"])
print("Top Predictive Network Flow Features (TreeSHAP):")
display(top_feats.head(7))

stability = xai.evaluate_explanation_stability_rbo(ton["X_test"][:300], n_splits=4)
print(f"
Explanation Stability (Rank-Biased Overlap): {stability['mean_rbo_stability']:.4f} (H4 Confirmed: {stability['is_stable_h4']})")
